# 🤖 Qwen3-TTS 1.7B - Autonomous Batch Runner (Headless Mode)
### Automated Dialogue Batch Synthesis for AI Agents (Antigravity)
* **Mode:** **Autonomous Headless Batch Runner**
* **Features:** Per-dialogue fault tolerance, immediate WAV file persistence, -14 LUFS broadcast mastering, and strict auto-termination (`sys.exit(0)`).


In [ ]:
# [CONFIGURATION] Antigravity Batch Dialogue Definitions
# Customize this list programmatically before dispatching.

BATCH_DIALOGUES = [
    {
        "id": "dialogue_001_villain",
        "mode": "design",
        "voice_description": "A chilling, menacing older villain, deep gravelly baritone with heavy pauses and cold authority.",
        "text": "Tumhe lagta hai tum mujhe rok sakte ho? ... Iss shahar ki har saans, mere ishaaron par chalti hai."
    },
    {
        "id": "dialogue_002_emotional",
        "mode": "design",
        "voice_description": "A sorrowful, tearful young woman, heartbroken whisper with trembling voice and breath intakes.",
        "text": "Maine tumpar sabse zyada bharosa kiya tha... Lekin tumne ek pal mein sab khatam kar diya."
    },
    {
        "id": "dialogue_003_host",
        "mode": "design",
        "voice_description": "A charismatic, warm and articulate professional host, fast and engaging conversational tempo.",
        "text": "Welcome back everyone! Today we are testing high-fidelity text-to-speech models running entirely on Kaggle cloud GPUs."
    }
]

print(f"Loaded {len(BATCH_DIALOGUES)} batch dialogue jobs.", flush=True)


In [ ]:
# [Step 1/3] Install Dependencies
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

subprocess.run("apt-get update -qq && apt-get install -y -qq sox libsox-fmt-all > /dev/null 2>&1", shell=True)
pkgs = ["transformers==4.57.3", "accelerate>=1.12.0", "soundfile", "pyloudnorm", "qwen-tts"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✓ Environment ready.", flush=True)


In [ ]:
# [Step 2/3] Load Models & Mastering Engine
import os, time, warnings, logging, io, contextlib
import numpy as np, soundfile as sf, torch, pyloudnorm as pyln

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from qwen_tts import Qwen3TTSModel

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

t0 = time.time()
design_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
print(f"✓ VoiceDesign model ready in {time.time() - t0:.1f}s", flush=True)

def master_audio(wav, sr, target_lufs=-14.0):
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.squeeze()
    wav = wav - np.mean(wav)
    try:
        meter = pyln.Meter(sr)
        current_lufs = meter.integrated_loudness(wav)
        if not np.isneginf(current_lufs) and not np.isnan(current_lufs):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                wav = pyln.normalize.loudness(wav, current_lufs, target_lufs)
    except Exception:
        pass
    peak = np.max(np.abs(wav))
    if peak > 0.95:
        wav = wav * (0.95 / peak)
    return wav


In [ ]:
# [Step 3/3] Fault-Tolerant Batch Execution & Immediate Auto-Termination
import time

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = []
print("=" * 75, flush=True)
print(f">>> Executing Batch Dialogue Queue: {len(BATCH_DIALOGUES)} Items", flush=True)
print("=" * 75, flush=True)

for i, job in enumerate(BATCH_DIALOGUES, 1):
    job_id = job.get("id", f"dialogue_{i}")
    out_file = os.path.join(OUTPUT_DIR, f"{job_id}.wav")
    print(f"\n[{i}/{len(BATCH_DIALOGUES)}] Synthesizing: {job_id}...", flush=True)
    t_start = time.time()
    
    # Fault tolerance: single error never halts the batch
    try:
        prompt = job.get("voice_description", "natural clear voice")
        wav, sr = design_model.generate_from_description(job["text"], prompt)
        mastered = master_audio(wav, sr)
        sf.write(out_file, mastered, sr, subtype="PCM_16")
        
        dur = len(mastered) / sr
        elapsed = time.time() - t_start
        print(f"✓ [{job_id}] Rendered in {elapsed:.1f}s ({dur:.1f}s audio) -> {out_file}", flush=True)
        results.append({"id": job_id, "status": "SUCCESS", "elapsed": f"{elapsed:.1f}s", "path": out_file})
    except Exception as e:
        elapsed = time.time() - t_start
        print(f"❌ [{job_id}] Error: {e} (Continuing to next dialogue...)", flush=True)
        results.append({"id": job_id, "status": "FAILED", "elapsed": f"{elapsed:.1f}s", "error": str(e)})

print("\n" + "=" * 80, flush=True)
print("🏁 BATCH SYNTHESIS COMPLETED! SUMMARY:", flush=True)
print("=" * 80, flush=True)
for r in results:
    icon = "✓" if r["status"] == "SUCCESS" else "❌"
    print(f" {icon} {r['id']} ({r['status']} in {r['elapsed']}) -> {r.get('path', r.get('error'))}", flush=True)
print("=" * 80, flush=True)

print("\n>>> Batch finished. Auto-terminating kernel to preserve Kaggle GPU quota. Goodbye!", flush=True)
time.sleep(1)
sys.exit(0)
